<a href="https://colab.research.google.com/github/Anushadhirde/Urban-Heat-Island-Change-Detection/blob/main/stack_bands.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
"""
STEP: Stack single-band TIFFs into multi-band TIFFs for ALL years/seasons.

WHAT CHANGED FROM THE SINGLE-FILE VERSION:
- The stacking logic is now wrapped in a function, stack_one_scene()
- A loop calls that function once per year/season combination
- Static layers (Elevation, Slope, Aspect) are read ONCE, not per year, then reused
- If a file is missing for a given year/season, it prints a warning and skips
  that one instead of crashing the whole run

BEFORE YOU RUN:
- Edit FOLDER, YEARS, SEASONS, and the filename pattern in build_path() below
  to match your actual PROCESSED_30M filenames.
- Install rasterio if you don't have it:  pip install rasterio
"""

import os
import rasterio
import numpy as np

# ---- 1. EDIT THESE to match your setup ----
FOLDER = "/content/drive/MyDrive/DATASET/PROCESSED_30M"
OUTPUT_FOLDER = "/content/drive/MyDrive/DATASET/STACKED"
YEARS = list(range(2017, 2025))     # edit to match the years you actually have
SEASONS = ["Summer", "Winter"]      # capitalized to match your filenames

DYNAMIC_VARS = ["LST", "NDVI", "NDBI", "NDWI", "Albedo"]
STATIC_VARS = ["Elevation", "Slope", "Aspect"]


def build_path(var_name, year=None, season=None):
    """
    Builds the expected filename for a variable.

    Dynamic (per year/season), e.g.:
      /content/drive/MyDrive/DATASET/PROCESSED_30M/LST/summer/LST_2017_Summer_Nagpur_30m.tif

    Static (no year/season), e.g.:
      /content/drive/MyDrive/DATASET/PROCESSED_30M/Aspect_Nagpur_30m.tif
    """
    if year is None:
        # static layers: directly in PROCESSED_30M, no subfolder
        return os.path.join(FOLDER, f"{var_name}_Nagpur_30m.tif")
    season_folder = season.lower()  # "Summer" -> "summer" for the folder name
    return os.path.join(
        FOLDER, var_name, season_folder, f"{var_name}_{year}_{season}_Nagpur_30m.tif"
    )


def load_static_bands():
    """Read the terrain layers once since they don't change across years."""
    static_bands = {}
    for var in STATIC_VARS:
        path = build_path(var)
        with rasterio.open(path) as src:
            static_bands[var] = src.read(1)
    return static_bands


def stack_one_scene(year, season, static_bands, band_order, meta_template, height, width):
    """Stack one year/season's dynamic layers together with the static layers."""
    bands = []
    for var in DYNAMIC_VARS:
        path = build_path(var, year, season)
        if not os.path.exists(path):
            print(f"  MISSING: {path} — skipping {year} {season}")
            return None
        with rasterio.open(path) as src:
            data = src.read(1)
        if data.shape != (height, width):
            print(f"  SHAPE MISMATCH: {path} is {data.shape}, expected {(height, width)} — skipping")
            return None
        bands.append(data)

    for var in STATIC_VARS:
        bands.append(static_bands[var])

    stacked = np.stack(bands, axis=0)

    meta = meta_template.copy()
    meta.update(count=len(band_order), dtype=stacked.dtype)

    out_path = os.path.join(OUTPUT_FOLDER, f"stack_{year}_{season}.tif")
    with rasterio.open(out_path, "w", **meta) as dst:
        for i, name in enumerate(band_order, start=1):
            dst.write(stacked[i - 1], i)
            dst.set_band_description(i, name)

    return out_path


def main():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    band_order = DYNAMIC_VARS + STATIC_VARS

    # Use the first dynamic file we find to establish the grid shape/metadata
    first_path = build_path(DYNAMIC_VARS[0], YEARS[0], SEASONS[0])
    with rasterio.open(first_path) as src:
        meta_template = src.meta.copy()
        height, width = src.height, src.width

    static_bands = load_static_bands()

    successes, failures = [], []
    for year in YEARS:
        for season in SEASONS:
            print(f"Processing {year} {season}...")
            result = stack_one_scene(
                year, season, static_bands, band_order, meta_template, height, width
            )
            if result:
                successes.append(result)
            else:
                failures.append((year, season))

    print("\n--- DONE ---")
    print(f"Successfully stacked: {len(successes)}")
    print(f"Skipped/failed: {len(failures)}")
    if failures:
        print("Missing or mismatched year/season combos:", failures)
    print(f"Band order in every output file: {band_order}")


if __name__ == "__main__":
    main()

Processing 2017 Summer...
  MISSING: /content/drive/MyDrive/DATASET/PROCESSED_30M/Albedo/summer/Albedo_2017_Summer_Nagpur_30m.tif — skipping 2017 Summer
Processing 2017 Winter...
  MISSING: /content/drive/MyDrive/DATASET/PROCESSED_30M/Albedo/winter/Albedo_2017_Winter_Nagpur_30m.tif — skipping 2017 Winter
Processing 2018 Summer...
  MISSING: /content/drive/MyDrive/DATASET/PROCESSED_30M/Albedo/summer/Albedo_2018_Summer_Nagpur_30m.tif — skipping 2018 Summer
Processing 2018 Winter...
  MISSING: /content/drive/MyDrive/DATASET/PROCESSED_30M/Albedo/winter/Albedo_2018_Winter_Nagpur_30m.tif — skipping 2018 Winter
Processing 2019 Summer...
  MISSING: /content/drive/MyDrive/DATASET/PROCESSED_30M/Albedo/summer/Albedo_2019_Summer_Nagpur_30m.tif — skipping 2019 Summer
Processing 2019 Winter...
  MISSING: /content/drive/MyDrive/DATASET/PROCESSED_30M/Albedo/winter/Albedo_2019_Winter_Nagpur_30m.tif — skipping 2019 Winter
Processing 2020 Summer...
  MISSING: /content/drive/MyDrive/DATASET/PROCESSED_30M/